# Fantasy Scoring Engine
Builds fantasy points from nflverse data, then checks them. Scoring settings reflect standard PPR league, specified explicitly in the setup cell below.

In [1]:
import numpy as np, pandas as pd
pd.set_option("display.max_columns", 60)

SCORING = {
    "passing_yards": 0.04,
    "passing_tds": 4,
    "interceptions": -2,
    "rushing_yards": 0.1,
    "rushing_tds": 6,
    "receptions": 1,
    "receiving_yards": 0.1,
    "receiving_tds": 6,
    "two_pt_conversions": 2,
    "fumbles_lost": -2,
    "special_teams_tds": 6,
}

MATCH_TOLERANCE = 0.1

pbp = pd.read_parquet("data/pbp_2026.parquet")
official = pd.read_parquet("data/player_stats_2026.parquet")

pbp = pbp[pbp["season_type"] == "REG"].copy()
official = official[official["season_type"] == "REG"].copy()

print(f"Plays loaded: {len(pbp):,}")
print(f"Official player-weeks loaded: {len(official):,}")

Plays loaded: 28,467
Official player-weeks loaded: 11,339


## Stat Ledger
I use a stat ledger instead of summing columns straight into a player table for retraceability. Once penalty-nullified plays are added back, the ledger makes it easy to still trace any point total to the exact plays that built it. 

In [2]:
KEYS = ["season", "week", "game_id", "play_id"]

def ledger_rows(df, player_col, stat_name, value):
    """Turn a set of plays into ledger rows for one stat.
    Value can be a column name or a number."""
    out = df[KEYS + [player_col]].rename(columns = {player_col: "player_id"})
    out["stat"] = stat_name
    out["value"] = df[value].values if isinstance(value, str) else value
    return out.dropna(subset=["player_id"])

# 2pa are scored separately, so they are split out from normal plays
is_2pt = pbp["two_point_attempt"] == 1
plays = pbp[~is_2pt]
two_pt = pbp[is_2pt & (pbp["two_point_conv_result"] == "success")]

In [3]:
# --------- passing ---------
pass_yds = ledger_rows(plays[plays["passing_yards"].notna()], "passer_player_id", "passing_yards", "passing_yards")
pass_td = ledger_rows(plays[plays["pass_touchdown"] == 1], "passer_player_id", "passing_tds", 1)
ints = ledger_rows(plays[plays["interception"] == 1], "passer_player_id", "interceptions", 1)

# --------- rushing ---------
rush_yds = ledger_rows(plays[plays["rushing_yards"].notna()], "rusher_player_id", "rushing_yards", "rushing_yards")
rush_td = ledger_rows(plays[plays["rush_touchdown"] == 1], "td_player_id", "rushing_tds", 1)

# --------- receiving ---------
recs = ledger_rows(plays[plays["complete_pass"] == 1], "receiver_player_id", "receptions", 1)
rec_yds = ledger_rows(plays[plays["receiving_yards"].notna()], "receiver_player_id", "receiving_yards", "receiving_yards")
rec_td = ledger_rows(plays[plays["pass_touchdown"] == 1], "td_player_id", "receiving_tds", 1)

# --------- laterals ---------
lat_rec = plays[plays["lateral_receiving_yards"].fillna(0) != 0]
lat_rush = plays[plays["lateral_rushing_yards"].fillna(0) != 0]
lateral_parts = [
    ledger_rows(lat_rec, "lateral_receiver_player_id", "receiving_yards", "lateral_receiving_yards"),
    ledger_rows(lat_rush, "lateral_rusher_player_id", "rushing_yards", "lateral_rushing_yards"),
]

# --------- two-point ---------
two_pt_rows = pd.concat([
    ledger_rows(two_pt, "passer_player_id", "two_pt_conversions", 1),
    ledger_rows(two_pt, "receiver_player_id", "two_pt_conversions", 1),
    ledger_rows(two_pt, "rusher_player_id", "two_pt_conversions", 1),
])

# --------- special teams TD ---------
st_parts = []
st = pbp[(pbp["return_touchdown"] == 1)
    & (pbp["play_type"].isin(["kickoff", "punt"]))
    & (pbp["td_team"] == pbp["return_team"])]
st_parts.append(ledger_rows(st, "td_player_id", "special_teams_tds", 1))

## Fumbles
A single play could have multiple fumbles, though rare, so each fumble is checked on its own. A fumble only counts as negative when the fumble is a turnover, not when recovered by a teammate. I account for 2 fumbles in one play, any more is extremely rare and unlikely to occur this season. The following loop will run twice for each fumble on a play, then applies.

In [4]:
fumble_parts = []
for n in [1, 2]:
    f = pbp[pbp[f"fumbled_{n}_player_id"].notna()].copy()
    no_recovery_turnover = f[f"fumble_recovery_{n}_team"].isna() & (f["fumble_lost"] == 1) & f["desc"].str.contains("Touchback", na=False)
    lost = (f[f"fumble_recovery_{n}_team"].notna() & (f[f"fumble_recovery_{n}_team"] != f[f"fumbled_{n}_team"])) | no_recovery_turnover
    f = f[lost]
    f = f[(f[f"fumbled_{n}_team"] == f["posteam"]) & (f["special_teams_play"] != 1)]
    fumble_parts.append(ledger_rows(f, f"fumbled_{n}_player_id", "fumbles_lost", 1))

In [5]:
ledger = pd.concat(
    [pass_yds, pass_td, ints, rush_yds, rush_td, recs, rec_yds, rec_td, two_pt_rows]
    + lateral_parts + fumble_parts + st_parts,
    ignore_index = True,
)
ledger["points"] = ledger["value"] * ledger["stat"].map(SCORING)
print(f"Ledger rows: {len(ledger):,}")
ledger.head()

Ledger rows: 31,117


,season,week,game_id,play_id,player_id,stat,value,points
0,2026,1,2026_01_ARI_LAC,62.0,00-0033119,passing_yards,27.0,1.08
1,2026,1,2026_01_ARI_LAC,154.0,00-0033119,passing_yards,5.0,0.20
2,2026,1,2026_01_ARI_LAC,247.0,00-0033119,passing_yards,11.0,0.44
3,2026,1,2026_01_ARI_LAC,386.0,00-0036355,passing_yards,18.0,0.72
4,2026,1,2026_01_ARI_LAC,433.0,00-0036355,passing_yards,23.0,0.92


## Player-week totals
The ledger is pivoted so each row is one player in one game, with a column for every stat and a final fantasy points column.

In [6]:
weekly_stats = (
    ledger.pivot_table(index=["season", "week", "game_id", "player_id"],
                       columns = "stat", values = "value", aggfunc = "sum", fill_value = 0)
    .reset_index()
)
weekly_points = (
    ledger.groupby(["season", "week", "game_id", "player_id"], as_index = False)["points"]
    .sum()
    .rename(columns = {"points": "fps"})
)
weekly = weekly_stats.merge(weekly_points, on = ["season", "week", "game_id", "player_id"])

# names and positions from official table for labeling
names = official[["player_id", "player_display_name", "position", "position_group"]].drop_duplicates("player_id")
weekly = weekly.merge(names, on = "player_id", how = "left")

weekly = weekly[weekly["position_group"].isin(["QB", "RB", "WR", "TE"])]
weekly.sort_values("fps", ascending = False).head()

,season,week,game_id,player_id,fumbles_lost,interceptions,passing_tds,passing_yards,receiving_tds,receiving_yards,receptions,rushing_tds,rushing_yards,special_teams_tds,two_pt_conversions,fps,player_display_name,position,position_group
2693,2026,10,2026_10_SF_DAL,00-0033280,0.0,0.0,0.0,0.0,2.0,161.0,4.0,1.0,62.0,0.0,1.0,46.3,Christian McCaffrey,RB,RB
967,2026,4,2026_04_DET_CAR,00-0040124,0.0,0.0,0.0,0.0,2.0,192.0,14.0,0.0,0.0,0.0,0.0,45.2,Tetairoa McMillan,WR,WR
2644,2026,10,2026_10_NE_DET,00-0039139,0.0,0.0,0.0,0.0,0.0,99.0,7.0,2.0,153.0,0.0,0.0,44.2,Jahmyr Gibbs,RB,RB
2698,2026,10,2026_10_SF_DAL,00-0037247,0.0,0.0,0.0,0.0,3.0,145.0,11.0,0.0,0.0,0.0,0.0,43.5,George Pickens,WR,WR
562,2026,2,2026_02_SEA_ARI,00-0038543,0.0,0.0,0.0,0.0,3.0,155.0,9.0,0.0,0.0,0.0,0.0,42.5,Jaxon Smith-Njigba,WR,WR


## Test 1: Do my points match official numbers?
Each player-week compared to official `fantasy_points_ppr` (nflverse).

In [7]:
official_skill = official[official["position"].isin(weekly["position"].unique())]
check = weekly.merge(
    official_skill[["player_id", "game_id", "fantasy_points_ppr"]],
    on = ["player_id", "game_id"], how = "outer", indicator = True,
)
check["diff"] = check["fps"].fillna(0) - check["fantasy_points_ppr"].fillna(0)
check["match"] = check["diff"].abs() <= MATCH_TOLERANCE

both = check[check["_merge"] == "both"]
print(f"Player-weeks in both tables: {len(both):,}")
print(f"Match rate: {both["match"].mean():.2%}")
print(f"Only in my table: {(check["_merge"] == "left_only").sum():,}")
print(f"Only in official table: {(check["_merge"] == "right_only").sum():,}")

Player-weeks in both tables: 2,955
Match rate: 99.93%
Only in my table: 0
Only in official table: 740


In [8]:
# mismatches
mismatches = both[~both["match"]].sort_values("diff", key = abs, ascending = False)
mismatches[["season", "week", "player_display_name", "position", "fps", "fantasy_points_ppr", "diff"]].head(20)

,season,week,player_display_name,position,fps,fantasy_points_ppr,diff
1401,2026.0,9.0,Rachaad White,RB,6.9,8.9,-2.0
2391,2026.0,8.0,Jalen Coker,WR,9.6,11.6,-2.0


In [9]:
# Stat-by-stat comparison shows which stat is causing a mismatch
STAT_MAP = {  # my column -> official column(s)
    "passing_yards": ["passing_yards"],
    "passing_tds": ["passing_tds"],
    "interceptions": ["passing_interceptions"],
    "rushing_yards": ["rushing_yards"],
    "rushing_tds": ["rushing_tds"],
    "receptions": ["receptions"],
    "receiving_yards": ["receiving_yards"],
    "receiving_tds": ["receiving_tds"],
    "two_pt_conversions": ["passing_2pt_conversions", "rushing_2pt_conversions", "receiving_2pt_conversions"],
    "fumbles_lost": ["sack_fumbles_lost", "rushing_fumbles_lost", "receiving_fumbles_lost"],
    "special_teams_tds": ["special_teams_tds"],
}
off_cols = sorted({c for cols in STAT_MAP.values() for c in cols})
# Official columns get an "off_" prefix so they don't collide with my columns of the same name
official_renamed = official[["player_id", "game_id"] + off_cols].rename(columns={c: "off_" + c for c in off_cols})
stat_check = weekly.merge(official_renamed, on=["player_id", "game_id"])

summary = []
for mine, theirs in STAT_MAP.items():
    my_vals = stat_check[mine] if mine in stat_check else 0
    their_vals = stat_check[["off_" + c for c in theirs]].fillna(0).sum(axis=1)
    summary.append({"stat": mine, "player_weeks_off": int(((my_vals - their_vals).abs() > 0.01).sum())})
pd.DataFrame(summary)

,stat,player_weeks_off
0,passing_yards,0
1,passing_tds,0
2,interceptions,0
3,rushing_yards,0
4,rushing_tds,0
5,receptions,0
6,receiving_yards,0
7,receiving_tds,0
8,two_pt_conversions,0
9,fumbles_lost,2


In [10]:
def trace(player_id, game_id):
    rows = ledger[(ledger["player_id"] == player_id) & (ledger["game_id"] == game_id)]
    return rows.merge(pbp[["game_id", "play_id", "desc"]], on=["game_id", "play_id"], how="left")

if len(mismatches):
    first = mismatches.iloc[0]
    display(trace(first["player_id"], first["game_id"]))

,season,week,game_id,play_id,player_id,stat,value,points,desc
0,2026,9,2026_09_LA_WAS,63.0,00-0037256,rushing_yards,3.0,0.3,(14:56) 1-R.White left end to WAS 36 for 3 yar...
1,2026,9,2026_09_LA_WAS,85.0,00-0037256,rushing_yards,3.0,0.3,(14:14) 1-R.White left end to WAS 39 for 3 yar...
2,2026,9,2026_09_LA_WAS,243.0,00-0037256,rushing_yards,5.0,0.5,(12:13) 1-R.White right tackle to LA 32 for 5 ...
3,2026,9,2026_09_LA_WAS,265.0,00-0037256,rushing_yards,2.0,0.2,(11:33) (Shotgun) 1-R.White up the middle to L...
4,2026,9,2026_09_LA_WAS,1452.0,00-0037256,rushing_yards,14.0,1.4,(9:02) (Shotgun) 1-R.White right guard to WAS ...
5,2026,9,2026_09_LA_WAS,1481.0,00-0037256,rushing_yards,-1.0,-0.1,(8:38) (No Huddle) 1-R.White right end to WAS ...
6,2026,9,2026_09_LA_WAS,2540.0,00-0037256,rushing_yards,2.0,0.2,(11:29) 1-R.White up the middle to WAS 22 for ...
7,2026,9,2026_09_LA_WAS,1000.0,00-0037256,receptions,1.0,1.0,(15:00) 5-J.Daniels pass short right to 1-R.Wh...
8,2026,9,2026_09_LA_WAS,3101.0,00-0037256,receptions,1.0,1.0,(4:55) (Shotgun) 5-J.Daniels pass short left t...
9,2026,9,2026_09_LA_WAS,1000.0,00-0037256,receiving_yards,45.0,4.5,(15:00) 5-J.Daniels pass short right to 1-R.Wh...


In [11]:
ledger.to_parquet("data/ledger_2026.parquet", index = False)
weekly.to_parquet("data/weekly_points_2026.parquet", index = False)